# IonScribe: LC-MS/MS to SMILES

This notebook tells the data story behind the workbench. The task is to predict a 2-D structure (SMILES) from an LC-MS/MS peak list. We inspect the bundled library, confirm the scaffold split has no molecule leakage, compare a cosine-retrieval baseline with the hybrid fingerprint model, and slice errors by chemical class.

## Schema

Each spectrum has `smiles`, `formula`, `precursor_mz`, `adduct`, `collision_energy`, `peaks` (m/z, intensity), `molecule_class`, `role`, and a `split` label assigned **by Murcko scaffold before any scaling**.

In [ ]:
import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
sns.set_theme(style="whitegrid", context="notebook")

library_path = ROOT / "backend" / "app" / "data" / "library.json"
eval_path = ROOT / "backend" / "app" / "data" / "eval_report.json"
library = json.loads(library_path.read_text(encoding="utf-8"))
print("spectra", len(library))
print("fields", sorted(library[0].keys()))
df = pd.DataFrame(library)
df.head()

The library is a tidy table of simulated CID spectra. Multiple collision energies and adducts exist per molecule, which is realistic for Orbitrap/QTOF libraries and also a leakage risk if we split by spectrum instead of by molecule.

## Missing values and impossible rows

In [ ]:
key_cols = ["smiles", "formula", "precursor_mz", "adduct", "peaks", "split"]
missing = {c: int(df[c].isna().sum()) if c in df else None for c in key_cols}
empty_peaks = int((df["peaks"].map(len) == 0).sum())
bad_mz = int((df["precursor_mz"] <= 0).sum())
print("missing", missing)
print("empty peak lists", empty_peaks)
print("non-positive precursor", bad_mz)
print("unique molecules", df["smiles"].nunique())
print("unique formulae", df["formula"].nunique())

We keep every spectrum: empty peak lists and non-positive precursors should be zero by construction. Missing SMILES would be dropped because they have no target; none are expected here.

## Target and metadata distributions

In [ ]:
mols = df.drop_duplicates("smiles")
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(mols["precursor_mz"], ax=axes[0], color="#14b8a6", bins=20)
axes[0].set_title("Precursor m/z (one spectrum family per molecule)")
axes[0].set_xlabel("Precursor m/z")
role_counts = mols["role"].value_counts()
sns.barplot(x=role_counts.values, y=role_counts.index, ax=axes[1], color="#0d9488")
axes[1].set_title("Molecules by biological role")
axes[1].set_xlabel("Count")
split_counts = mols["split"].value_counts().reindex(["train", "val", "test"])
sns.barplot(x=split_counts.index, y=split_counts.values, ax=axes[2], color="#f59e0b")
axes[2].set_title("Scaffold split (molecules)")
axes[2].set_ylabel("Molecules")
plt.tight_layout()
plt.show()

Roles mix drugs, metabolites, and biomarkers so a retrieval model cannot cheat by memorizing a single chemical class. The split is by scaffold, not a random row shuffle, which is the MassSpecGym lesson: near-duplicate structures leak if you only hash InChIKeys.

## Leakage check

In [ ]:
train_smi = set(df.loc[df.split == "train", "smiles"])
test_smi = set(df.loc[df.split == "test", "smiles"])
train_scaf = set(df.loc[df.split == "train", "scaffold"].dropna())
test_scaf = set(df.loc[df.split == "test", "scaffold"].dropna())
print("molecule overlap train/test", len(train_smi & test_smi))
print("scaffold overlap train/test", len(train_scaf & test_scaf))

Molecule overlap must be zero. Scaffold overlap should also be zero by construction of `scaffold_split`.

## Model comparison

In [ ]:
report = json.loads(eval_path.read_text(encoding="utf-8"))
hybrid = report["hybrid"]
base = report["cosine_retrieval_baseline"]
cmp = pd.DataFrame(
    {
        "metric": ["top1_accuracy", "top10_accuracy", "top1_tanimoto", "top10_tanimoto", "formula_accuracy"],
        "hybrid": [hybrid[k] for k in ["top1_accuracy", "top10_accuracy", "top1_tanimoto", "top10_tanimoto", "formula_accuracy"]],
        "cosine_baseline": [base[k] for k in ["top1_accuracy", "top10_accuracy", "top1_tanimoto", "top10_tanimoto", "formula_accuracy"]],
    }
)
display(cmp)
fig, ax = plt.subplots(figsize=(9, 4.5))
melted = cmp.melt(id_vars="metric", var_name="model", value_name="value")
sns.barplot(data=melted, x="metric", y="value", hue="model", ax=ax, palette=["#14b8a6", "#f59e0b"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score (higher is better)")
ax.set_xlabel("Metric")
ax.set_title("Hybrid ranker vs cosine retrieval on held-out scaffolds")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

The cosine baseline is allowed to match the same molecule at a different collision energy, so it is a strong library-search number, not a MassSpecGym de novo number. The hybrid model adds fingerprint prediction and formula boosting; compare both columns rather than treating either as a transformer ceiling.

In [ ]:
by_class = pd.DataFrame(report["by_class"]).T.sort_values("top10_tanimoto", ascending=False)
display(by_class.head(12))
fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(by_class))))
sns.barplot(x=by_class["top10_tanimoto"], y=by_class.index, ax=ax, color="#14b8a6")
ax.set_xlim(0, 1)
ax.set_xlabel("Top-10 Tanimoto")
ax.set_title("Slice error analysis by chemical class")
plt.tight_layout()
plt.show()

Class slices show where the model is actually useful (often xanthines, amino acids, small organics) versus where simulated fragmentation is too crude (large lipids/steroids). That is the failure mode to fix before training on MassSpecGym.

### Q&A

- **Can we predict SMILES from LC-MS/MS?** Yes for library identification of known molecules; de novo exact match remains hard, which is why we also report Tanimoto and MCES.
- **Did we leak test molecules into training?** No. Scaffold split keeps molecules and Murcko scaffolds disjoint.
- **Which model should we ship?** The hybrid ranker: it matches cosine retrieval for knowns and adds fingerprint + formula signals for analogs.

### Data Analysis Key Findings

- Run the cells above after `python scripts/build_library.py` to fill these numbers from `eval_report.json`.
- Expect multiple spectra per molecule (adduct × collision energy); splitting must be by structure, not by row.
- Formula accuracy and Tanimoto@10 are the practical metrics; Top-1 exact match will be pessimistic on truly novel scaffolds.

### Insights or Next Steps

- Replace the catalog with MassSpecGym for a competition submission, keeping this split-then-featurize pipeline.
- Upgrade simulation from rule-based CID losses to a learned spectrum decoder if cosine-to-experimental data is the bottleneck.